In [22]:
import pandas as pd
import torch
from torch_geometric.transforms import RandomLinkSplit
from sklearn.manifold import TSNE
import plotly.express as px
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



from train_model_DESC import run_training, LM, lmbda

In [29]:
def get_node_embeddings(model, data):
    model.eval()
    with torch.no_grad():
        node_embeddings = model.encode(data.x, data.edge_index)
    node_embeddings = node_embeddings.cpu().numpy()
    return node_embeddings

def get_reduced_embeddings(node_embeddings):
    # Initialize t-SNE with 2 components for 2D visualization
    tsne = TSNE(n_components=2 ) #, random_state=42) # 42

    # Apply t-SNE to the drug embeddings
    reduced_embeddings = tsne.fit_transform(node_embeddings)

    return pd.DataFrame(reduced_embeddings, columns=['TSNE-1', 'TSNE-2'])

def plot_embeddings(reduced_embeddings_df):
    # Create a scatter plot using Plotly
    fig = px.scatter(
        reduced_embeddings_df,
        x='TSNE-1',
        y='TSNE-2',
        title='t-SNE Visualization of Drug Embeddings',
        width=800,
        height=600
    )
    fig.show()
    #return fig

In [16]:
#def main():
models = {'GPT+Desc': '/data/giobbi/embeddings/Dr_Desc_GPT.csv'}    

# Data loading
DDI_graph = pd.read_csv('https://raw.githubusercontent.com/liiniix/BioSNAP/master/ChCh-Miner/ChCh-Miner_durgbank-chem-chem.tsv', sep='\t')
DDI_graph.rename(columns={'Drug1': 'src', 'Drug2': 'dst'}, inplace=True)

transform = RandomLinkSplit(
    num_val=0.2,
    num_test=0.2,
    is_undirected=True,
    add_negative_train_samples=False,
    neg_sampling_ratio=1.0
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 100
LR = [0.0003]

results = {}
for modelname, dir in models.items():
    current_graph = DDI_graph

    drugID_DESC = pd.read_csv(dir, sep='\t', index_col=0)
    allowed_drug = list(drugID_DESC['Drug ID'])

    emb = LM(current_graph, allowed_drug, dir)

    for lr in LR:
        print(f'======== {modelname} | LR: {lr} ========')
        model, data, metrics = run_training(emb, transform, device, lmbda, epochs=epochs, lr=lr)
        results[(modelname, lr)] = {'model': model, 'data': data, 'metrics': metrics}


for key, value in results.items():
    print(f"Model: {key[0]}, LR: {key[1]}, Metrics: {value['metrics']}")




======== GPT+Desc | LR: 0.0003 ========
-------------------------------
Training with LR: 0.0003
Epoch: 001, Loss: 0.7037, Val: 0.5781
Epoch: 002, Loss: 0.6979, Val: 0.5687
Epoch: 003, Loss: 0.6952, Val: 0.6031
Epoch: 004, Loss: 0.6940, Val: 0.6995
Epoch: 005, Loss: 0.6935, Val: 0.7587
Epoch: 006, Loss: 0.6932, Val: 0.8227
Epoch: 007, Loss: 0.6930, Val: 0.8466
Epoch: 008, Loss: 0.6929, Val: 0.8667
Epoch: 009, Loss: 0.6928, Val: 0.8868
Epoch: 010, Loss: 0.6926, Val: 0.8986
Epoch: 011, Loss: 0.6925, Val: 0.9235
Epoch: 012, Loss: 0.6922, Val: 0.9326
Epoch: 013, Loss: 0.6921, Val: 0.9429
Epoch: 014, Loss: 0.6919, Val: 0.9518
Epoch: 015, Loss: 0.6917, Val: 0.9577
Epoch: 016, Loss: 0.6914, Val: 0.9645
Epoch: 017, Loss: 0.6912, Val: 0.9675
Epoch: 018, Loss: 0.6908, Val: 0.9723
Epoch: 019, Loss: 0.6903, Val: 0.9723
Epoch: 020, Loss: 0.6900, Val: 0.9728
Epoch: 021, Loss: 0.6895, Val: 0.9760
Epoch: 022, Loss: 0.6890, Val: 0.9739
Epoch: 023, Loss: 0.6885, Val: 0.9766
Epoch: 024, Loss: 0.6878, Val

In [27]:
embedding = get_node_embeddings(model, data)
embedding = get_reduced_embeddings(embedding)

In [28]:
plot_embeddings(embedding)

In [30]:
# TODO: add labels to the plot using drugID_DESC -> Make sure the order of embeddings matches the order of drugID_DESC
# TODO: color by drug category if available